In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    confusion_matrix,
    accuracy_score,
    classification_report
)
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Load data
df = pd.read_csv("final_integrated_model2_classification.csv")

# 2. Define target and features
y = df["HighViolation"]
X = df.drop(columns=["HighViolation"])

# 3. Identify categorical columns
cat_cols = [c for c in ["Facility", "Direction"] if c in X.columns]
num_cols = [c for c in X.columns if c not in cat_cols]

# 4. Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

# 5. Closest sklearn replication of H2O GBM
model = GradientBoostingClassifier(
    n_estimators=514,
    learning_rate=0.1,
    max_depth=9,
    random_state=42
)

pipeline = Pipeline([
    ("pre", preprocessor),
    ("model", model)
])

# 6. Stratified 5-fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

aucs = []
loglosses = []
accuracies = []

all_true = []
all_pred = []
all_prob = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    pipeline.fit(X_train, y_train)

    prob = pipeline.predict_proba(X_test)[:, 1]
    pred = pipeline.predict(X_test)

    auc = roc_auc_score(y_test, prob)
    ll = log_loss(y_test, prob, labels=[0, 1])
    acc = accuracy_score(y_test, pred)

    aucs.append(auc)
    loglosses.append(ll)
    accuracies.append(acc)

    all_true.extend(y_test.tolist())
    all_pred.extend(pred.tolist())
    all_prob.extend(prob.tolist())

# 7. Final metrics
print("Mean CV AUC:", np.mean(aucs))
print("Mean CV LogLoss:", np.mean(loglosses))
print("Mean CV Accuracy:", np.mean(accuracies))

print("\nConfusion Matrix:")
print(confusion_matrix(all_true, all_pred))

print("\nClassification Report:")
print(classification_report(all_true, all_pred))

Mean CV AUC: 1.0
Mean CV LogLoss: 1.2313393247514042e-08
Mean CV Accuracy: 1.0

Confusion Matrix:
[[144   0]
 [  0  48]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       144
           1       1.00      1.00      1.00        48

    accuracy                           1.00       192
   macro avg       1.00      1.00      1.00       192
weighted avg       1.00      1.00      1.00       192

